## Voci – L'archivio dei contenuti
### Progetto Python - start2impact

**Voci** è un media brand che dà spazio alle storie e alle idee delle donne su blog, social e YouTube.

Anche nei media digitali le donne parlano poco: in Italia sono solo il **39% degli esperti citati** e firmano il **29% dei contenuti** (GMMP 2025). Voci non punta alla semplice parità, ma vuole **compensare** questa narrazione, dando alle donne più spazio del solito.

Questo notebook costruisce l'**archivio dei contenuti di Voci**: la **redazione** registra i contenuti, che entrano solo se rispettano la linea editoriale; gli **utenti** li consultano con ricerca, filtri e statistiche.

**Indice**
1. Le regole dell'archivio
2. Le classi
3. La redazione al lavoro
4. Ricerca e filtri
5. Statistiche
6. Ora tocca a te
7. Fonti

In [1]:
import pandas as pd

## 1. Le regole dell'archivio

Per ogni contenuto **contiamo le voci**, cioè le persone che parlano: intervistate, esperte, ospiti. Chi firma non entra nel conteggio.

**Scegliamo una soglia.** Nei media digitali le voci maschili tra gli esperti citati sono circa il 61%: Voci ribalta la proporzione, e **le voci maschili dell'intero archivio non possono superare il 30%**.

La soglia si calcola sull'archivio intero, compreso il nuovo contenuto: se resta entro il 30% il contenuto entra (anche se contiene voci maschili), altrimenti viene rifiutato con l'errore `VociSbilanciate`.

**Valori ammessi:** tipi video, intervista, articolo · canali blog, youtube, social

## 2. Le classi

### 2.1 `Contenuto` e le sue classi figlie
`Contenuto` è la classe base: titolo, tema, canale, data e un dizionario con le voci (`{"donne": n, "uomini": n}`). La data è un testo `"AAAA-MM-GG"`: in questo formato l'ordine alfabetico coincide con quello cronologico, quindi il contenuto più recente è il valore massimo.

`Video`, `Intervista` e `Articolo` ereditano da `Contenuto`, aggiungono un attributo ciascuna e fanno l'override di `__str__`.

In [2]:
class Contenuto:
    """Contenuto generico dell'archivio: titolo, tema, canale, data e voci."""

    def __init__(self, titolo, tema, canale, data, donne, uomini):
        self.titolo = titolo
        self.tema = tema.lower()
        self.canale = canale.lower()
        self.data = data
        self.voci = {"donne": donne, "uomini": uomini}
        self.tipo = "contenuto"

    def totale_voci(self):
        """Restituisce il numero totale di persone che parlano nel contenuto."""
        return self.voci["donne"] + self.voci["uomini"]

    def percentuale_donne(self):
        """Restituisce la percentuale di voci femminili del contenuto."""
        if self.totale_voci() == 0:
            return 0
        return self.voci["donne"] / self.totale_voci() * 100

    def __str__(self):
        """Restituisce il contenuto come testo leggibile."""
        return (f"[{self.tipo}] {self.titolo} | {self.tema} | {self.canale} | {self.data} | "
                f"voci femminili {self.percentuale_donne():.0f}% "
                f"(donne {self.voci['donne']}, uomini {self.voci['uomini']})")


class Video(Contenuto):
    """Video pubblicato da Voci: aggiunge la durata in minuti."""

    def __init__(self, titolo, tema, canale, data, donne, uomini, durata):
        super().__init__(titolo, tema, canale, data, donne, uomini)
        self.tipo = "video"
        self.durata = durata

    def __str__(self):
        """Aggiunge la durata alla descrizione del contenuto."""
        return super().__str__() + f" | durata {self.durata} min"


class Intervista(Contenuto):
    """Intervista: aggiunge il nome della persona intervistata."""

    def __init__(self, titolo, tema, canale, data, donne, uomini, intervistata):
        super().__init__(titolo, tema, canale, data, donne, uomini)
        self.tipo = "intervista"
        self.intervistata = intervistata

    def __str__(self):
        """Aggiunge la persona intervistata alla descrizione del contenuto."""
        return super().__str__() + f" | con {self.intervistata}"


class Articolo(Contenuto):
    """Articolo del blog: aggiunge la firma."""

    def __init__(self, titolo, tema, canale, data, donne, uomini, firma):
        super().__init__(titolo, tema, canale, data, donne, uomini)
        self.tipo = "articolo"
        self.firma = firma

    def __str__(self):
        """Aggiunge la firma alla descrizione del contenuto."""
        return super().__str__() + f" | di {self.firma}"

### 2.2 L'errore `VociSbilanciate`
È un'eccezione creata da noi: eredita da `Exception`, costruisce il messaggio con i dati del contenuto rifiutato e lo passa alla classe madre con `super().__init__()`.

In [3]:
class VociSbilanciate(Exception):
    """Errore sollevato quando un contenuto porterebbe le voci maschili dell'archivio oltre la soglia."""

    def __init__(self, titolo, percentuale_uomini, soglia):
        self.titolo = titolo
        self.percentuale_uomini = percentuale_uomini
        self.soglia = soglia
        messaggio = (f"'{titolo}' non è stato aggiunto: con questo contenuto le voci maschili "
                     f"dell'archivio salirebbero al {percentuale_uomini:.1f}%, oltre la soglia del {soglia}%.")
        super().__init__(messaggio)

### 2.3 `ArchivioVoci`
È il gestore della collezione: i contenuti sono salvati in un dizionario `{titolo: oggetto Contenuto}`.

| Metodo | Cosa fa |
|---|---|
| `aggiungi` | controlla dati e soglia, registra il contenuto e stampa la percentuale aggiornata di voci femminili |
| `rimuovi` | elimina un contenuto dato il titolo |
| `mostra` | stampa l'archivio, oppure una lista di risultati |
| `cerca` | ricerca base: una parola nel titolo o nel tema |
| `filtra` | filtraggio avanzato: più criteri insieme (tipo, canale, tema, percentuale minima di voci femminili) |
| `temi` | elenca i temi presenti nell'archivio |
| `statistiche` | conteggio, distribuzione per tipo, tema più trattato, contenuto più recente, tema su cui impegnarsi di più |

Il trattino basso di `_percentuale_uomini` indica un metodo di uso **interno** alla classe.

In [4]:
class ArchivioVoci:
    """Archivio dei contenuti di Voci: un dizionario {titolo: oggetto Contenuto}."""

    def __init__(self, soglia_uomini=30, riferimento_digitale=39):
        self.contenuti = {}
        self.soglia_uomini = soglia_uomini                  # % massima di voci maschili nell'archivio
        self.riferimento_digitale = riferimento_digitale    # % di esperte citate nei media digitali (GMMP 2025)
        self.tipi_ammessi = ["video", "intervista", "articolo"]
        self.canali_ammessi = ["blog", "youtube", "social"]

    def _percentuale_uomini(self, contenuti):
        """Calcola la percentuale di voci maschili su una lista di contenuti."""
        donne = 0
        uomini = 0
        for contenuto in contenuti:
            donne += contenuto.voci["donne"]
            uomini += contenuto.voci["uomini"]
        if donne + uomini == 0:
            return 0
        return uomini / (donne + uomini) * 100

    def aggiungi(self, contenuto):
        """Aggiunge un contenuto se i dati sono validi e l'archivio resta entro la soglia di voci maschili."""
        if contenuto.titolo.strip() == "":
            raise ValueError("il titolo non può essere vuoto")
        elif contenuto.titolo in self.contenuti:
            raise ValueError(f"'{contenuto.titolo}' è già presente nell'archivio")
        elif contenuto.voci["donne"] < 0 or contenuto.voci["uomini"] < 0:
            raise ValueError(f"'{contenuto.titolo}': il numero di voci non può essere negativo")
        elif contenuto.totale_voci() == 0:
            raise ValueError(f"'{contenuto.titolo}': un contenuto deve avere almeno una voce")

        contenuti_dopo = list(self.contenuti.values()) + [contenuto]
        percentuale_uomini = self._percentuale_uomini(contenuti_dopo)
        if percentuale_uomini > self.soglia_uomini:
            raise VociSbilanciate(contenuto.titolo, percentuale_uomini, self.soglia_uomini)

        self.contenuti[contenuto.titolo] = contenuto
        print(f"Aggiunto: '{contenuto.titolo}' → voci femminili nell'archivio: {100 - percentuale_uomini:.1f}% "
              f"(esperte nei media digitali: {self.riferimento_digitale}%)")

    def rimuovi(self, titolo):
        """Rimuove un contenuto dall'archivio dato il titolo."""
        try:
            del self.contenuti[titolo]
            print(f"Rimosso: '{titolo}'")
        except KeyError:
            print(f"Errore: '{titolo}' non è presente nell'archivio")

    def mostra(self, contenuti=None):
        """Stampa tutti i contenuti dell'archivio, oppure solo quelli della lista ricevuta."""
        if contenuti is None:
            contenuti = list(self.contenuti.values())
        if len(contenuti) == 0:
            print("Nessun contenuto da mostrare.")
            return
        for contenuto in contenuti:
            print(contenuto)

    def cerca(self, parola):
        """Ricerca base: restituisce i contenuti che hanno la parola nel titolo o nel tema."""
        parola = parola.strip().lower()
        if parola == "":
            raise ValueError("scrivi almeno una parola da cercare")
        risultati = []
        for contenuto in self.contenuti.values():
            if parola in contenuto.titolo.lower() or parola in contenuto.tema:
                risultati.append(contenuto)
        return risultati

    def filtra(self, tipo=None, canale=None, tema=None, min_donne=None):
        """Filtraggio avanzato: restituisce i contenuti che rispettano tutti i criteri indicati."""
        risultati = []
        for contenuto in self.contenuti.values():
            if tipo is not None and contenuto.tipo != tipo:
                continue                                # non rispetta il tipo: lo salto
            elif canale is not None and contenuto.canale != canale:
                continue
            elif tema is not None and contenuto.tema != tema:
                continue
            elif min_donne is not None and contenuto.percentuale_donne() < min_donne:
                continue
            else:
                risultati.append(contenuto)             # ha superato tutti i criteri
        return risultati

    def temi(self):
        """Restituisce la lista dei temi presenti nell'archivio, senza ripetizioni."""
        elenco = []
        for contenuto in self.contenuti.values():
            if contenuto.tema not in elenco:
                elenco.append(contenuto.tema)
        return sorted(elenco)

    def statistiche(self):
        """Stampa le statistiche dell'archivio usando una tabella pandas."""
        if len(self.contenuti) == 0:
            print("L'archivio è vuoto: nessuna statistica da calcolare.")
            return

        righe = []
        for contenuto in self.contenuti.values():
            righe.append({"titolo": contenuto.titolo, "tipo": contenuto.tipo, "canale": contenuto.canale,
                          "tema": contenuto.tema, "data": contenuto.data,
                          "donne": contenuto.voci["donne"], "uomini": contenuto.voci["uomini"]})
        tabella = pd.DataFrame(righe)

        donne_archivio = 100 - self._percentuale_uomini(self.contenuti.values())
        print(f"Contenuti in archivio: {len(tabella)}")
        print(f"Voci femminili nell'archivio: {donne_archivio:.1f}% "
              f"(esperte nei media digitali: {self.riferimento_digitale}%)")

        print("\nContenuti per tipo:")
        for tipo, percentuale in (tabella["tipo"].value_counts(normalize=True) * 100).items():
            print(f"  {tipo}: {percentuale:.0f}%")

        conteggio_temi = tabella["tema"].value_counts()
        print(f"\nTema più trattato: {conteggio_temi.index[0]} ({conteggio_temi.iloc[0]} contenuti)")

        piu_recente = tabella.sort_values("data").iloc[-1]
        print(f"Contenuto più recente: '{piu_recente['titolo']}' ({piu_recente['data']})")

        per_tema = tabella.groupby("tema")[["donne", "uomini"]].sum()
        per_tema["% donne"] = per_tema["donne"] / (per_tema["donne"] + per_tema["uomini"]) * 100
        tema_debole = per_tema["% donne"].idxmin()
        print(f"Tema su cui Voci deve impegnarsi di più: {tema_debole} "
              f"({per_tema.loc[tema_debole, '% donne']:.0f}% di voci femminili)")

## 3. La redazione al lavoro

### 3.1 Il primo archivio
La redazione registra sette contenuti a forte prevalenza femminile; ogni aggiunta stampa la percentuale aggiornata di voci femminili. Contenuti e persone sono inventati, i temi riprendono questioni reali del settore dell'informazione.

In [5]:
archivio = ArchivioVoci()

primi_contenuti = [
    Intervista("Madri e giornaliste: il rientro dopo il congedo", "lavoro e maternità", "youtube",
               "2026-09-14", 3, 0, "Chiara Lodi"),
    Video("La piramide delle redazioni", "leadership", "youtube", "2026-09-21", 4, 1, 12),
    Articolo("Chi firma le notizie in Italia", "nuovi media", "blog", "2026-09-28", 5, 1, "Marta Rinaldi"),
    Intervista("Fondare un media a trent'anni", "nuovi media", "youtube", "2026-10-05", 2, 0, "Sara Monti"),
    Articolo("Il gender pay gap spiegato in 5 numeri", "economia", "blog", "2026-10-08", 3, 1, "Elena Costa"),
    Video("Allenatrici in serie A", "sport", "social", "2026-10-12", 2, 3, 4),
    Video("Le esperte che i media non chiamano", "scienza e tecnologia", "youtube", "2026-10-22", 6, 0, 15),
]

for contenuto in primi_contenuti:
    try:
        archivio.aggiungi(contenuto)
    except (ValueError, VociSbilanciate) as errore:
        print(f"Errore: {errore}")

Aggiunto: 'Madri e giornaliste: il rientro dopo il congedo' → voci femminili nell'archivio: 100.0% (esperte nei media digitali: 39%)
Aggiunto: 'La piramide delle redazioni' → voci femminili nell'archivio: 87.5% (esperte nei media digitali: 39%)
Aggiunto: 'Chi firma le notizie in Italia' → voci femminili nell'archivio: 85.7% (esperte nei media digitali: 39%)
Aggiunto: 'Fondare un media a trent'anni' → voci femminili nell'archivio: 87.5% (esperte nei media digitali: 39%)
Aggiunto: 'Il gender pay gap spiegato in 5 numeri' → voci femminili nell'archivio: 85.0% (esperte nei media digitali: 39%)
Aggiunto: 'Allenatrici in serie A' → voci femminili nell'archivio: 76.0% (esperte nei media digitali: 39%)
Aggiunto: 'Le esperte che i media non chiamano' → voci femminili nell'archivio: 80.6% (esperte nei media digitali: 39%)


In [6]:
archivio.mostra()

[intervista] Madri e giornaliste: il rientro dopo il congedo | lavoro e maternità | youtube | 2026-09-14 | voci femminili 100% (donne 3, uomini 0) | con Chiara Lodi
[video] La piramide delle redazioni | leadership | youtube | 2026-09-21 | voci femminili 80% (donne 4, uomini 1) | durata 12 min
[articolo] Chi firma le notizie in Italia | nuovi media | blog | 2026-09-28 | voci femminili 83% (donne 5, uomini 1) | di Marta Rinaldi
[intervista] Fondare un media a trent'anni | nuovi media | youtube | 2026-10-05 | voci femminili 100% (donne 2, uomini 0) | con Sara Monti
[articolo] Il gender pay gap spiegato in 5 numeri | economia | blog | 2026-10-08 | voci femminili 75% (donne 3, uomini 1) | di Elena Costa
[video] Allenatrici in serie A | sport | social | 2026-10-12 | voci femminili 40% (donne 2, uomini 3) | durata 4 min
[video] Le esperte che i media non chiamano | scienza e tecnologia | youtube | 2026-10-22 | voci femminili 100% (donne 6, uomini 0) | durata 15 min


### 3.2 La regola della soglia in azione
- un'**intervista** con una sindaca e un assessore: l'archivio resta sotto il 30% e il contenuto **entra**;
- una **tavola rotonda** con nove economisti e una sola economista: l'archivio supererebbe il 30% e il contenuto **viene rifiutato**.

In [7]:
intervista_assessore = Intervista("Mobilità e città a misura di donna", "politica", "youtube",
                                  "2026-10-23", 1, 1, "Anna Bruni")
tavola_rotonda = Video("Tavola rotonda: dove va l'economia", "economia", "youtube",
                       "2026-10-24", 1, 9, 45)

for contenuto in [intervista_assessore, tavola_rotonda]:
    try:
        archivio.aggiungi(contenuto)
    except VociSbilanciate as errore:
        print(f"Rifiutato → {errore}")

Aggiunto: 'Mobilità e città a misura di donna' → voci femminili nell'archivio: 78.8% (esperte nei media digitali: 39%)
Rifiutato → 'Tavola rotonda: dove va l'economia' non è stato aggiunto: con questo contenuto le voci maschili dell'archivio salirebbero al 37.2%, oltre la soglia del 30%.


### 3.3 Dati non validi
Titoli già presenti e numeri di voci negativi o pari a zero vengono rifiutati prima della registrazione.

In [8]:
contenuti_errati = [
    Articolo("Chi firma le notizie in Italia", "nuovi media", "blog", "2026-10-24", 2, 0, "Marta Rinaldi"),
    Intervista("Donne e sindacato", "lavoro e maternità", "youtube", "2026-10-24", -1, 0, "Rita Sala"),
    Articolo("Editoriale senza voci", "nuovi media", "blog", "2026-10-24", 0, 0, "Redazione"),
]

for contenuto in contenuti_errati:
    try:
        archivio.aggiungi(contenuto)
    except (ValueError, VociSbilanciate) as errore:
        print(f"Errore: {errore}")

Errore: 'Chi firma le notizie in Italia' è già presente nell'archivio
Errore: 'Donne e sindacato': il numero di voci non può essere negativo
Errore: 'Editoriale senza voci': un contenuto deve avere almeno una voce


### 3.4 Rimuovere un contenuto

In [9]:
archivio.rimuovi("Il gender pay gap spiegato in 5 numeri")
archivio.rimuovi("Un titolo che non esiste")

Rimosso: 'Il gender pay gap spiegato in 5 numeri'
Errore: 'Un titolo che non esiste' non è presente nell'archivio


## 4. Ricerca e filtri

### 4.1 Ricerca base
Una parola, cercata nel titolo e nel tema, senza distinguere maiuscole e minuscole.

In [10]:
print("Ricerca: 'media'")
archivio.mostra(archivio.cerca("media"))

Ricerca: 'media'
[articolo] Chi firma le notizie in Italia | nuovi media | blog | 2026-09-28 | voci femminili 83% (donne 5, uomini 1) | di Marta Rinaldi
[intervista] Fondare un media a trent'anni | nuovi media | youtube | 2026-10-05 | voci femminili 100% (donne 2, uomini 0) | con Sara Monti
[video] Le esperte che i media non chiamano | scienza e tecnologia | youtube | 2026-10-22 | voci femminili 100% (donne 6, uomini 0) | durata 15 min


### 4.2 Filtraggio avanzato
Più criteri insieme, tutti facoltativi: un contenuto compare solo se li rispetta **tutti**.

In [11]:
print("Interviste pubblicate su YouTube:")
archivio.mostra(archivio.filtra(tipo="intervista", canale="youtube"))

print("\nContenuti su YouTube con almeno l'80% di voci femminili:")
archivio.mostra(archivio.filtra(canale="youtube", min_donne=80))

Interviste pubblicate su YouTube:
[intervista] Madri e giornaliste: il rientro dopo il congedo | lavoro e maternità | youtube | 2026-09-14 | voci femminili 100% (donne 3, uomini 0) | con Chiara Lodi
[intervista] Fondare un media a trent'anni | nuovi media | youtube | 2026-10-05 | voci femminili 100% (donne 2, uomini 0) | con Sara Monti
[intervista] Mobilità e città a misura di donna | politica | youtube | 2026-10-23 | voci femminili 50% (donne 1, uomini 1) | con Anna Bruni

Contenuti su YouTube con almeno l'80% di voci femminili:
[intervista] Madri e giornaliste: il rientro dopo il congedo | lavoro e maternità | youtube | 2026-09-14 | voci femminili 100% (donne 3, uomini 0) | con Chiara Lodi
[video] La piramide delle redazioni | leadership | youtube | 2026-09-21 | voci femminili 80% (donne 4, uomini 1) | durata 12 min
[intervista] Fondare un media a trent'anni | nuovi media | youtube | 2026-10-05 | voci femminili 100% (donne 2, uomini 0) | con Sara Monti
[video] Le esperte che i media 

## 5. Statistiche
Conteggio, distribuzione per tipo, tema più trattato, contenuto più recente e, soprattutto, **il tema su cui la redazione deve impegnarsi di più**, cioè quello con meno voci femminili.

In [12]:
archivio.statistiche()

Contenuti in archivio: 7
Voci femminili nell'archivio: 79.3% (esperte nei media digitali: 39%)

Contenuti per tipo:
  intervista: 43%
  video: 43%
  articolo: 14%

Tema più trattato: nuovi media (2 contenuti)
Contenuto più recente: 'Mobilità e città a misura di donna' (2026-10-23)
Tema su cui Voci deve impegnarsi di più: sport (40% di voci femminili)


## 6. Ora tocca a te
Adesso l'utente sei tu: dal menu puoi **sfogliare l'archivio, cercare una parola, filtrare per più criteri e vedere le statistiche**.

Ogni risposta viene controllata, quindi un valore sbagliato produce solo un messaggio e una nuova domanda. Nei filtri premi **Invio** per saltare un criterio; per uscire scegli **0**.

In [13]:
def chiedi_scelta(nome_criterio, opzioni):
    """Chiede un valore tra le opzioni ammesse; restituisce None se l'utente preme Invio."""
    while True:
        risposta = input(f"{nome_criterio} ({' / '.join(opzioni)} – Invio per saltare): ").strip().lower()
        if risposta == "":
            return None
        if risposta in opzioni:
            return risposta
        print(f"'{risposta}' non è un valore valido. Scegli tra: {', '.join(opzioni)}")


def chiedi_percentuale():
    """Chiede la percentuale minima di voci femminili; restituisce None se l'utente preme Invio."""
    while True:
        risposta = input("Percentuale minima di voci femminili (0-100 – Invio per saltare): ")
        risposta = risposta.strip().replace("%", "").replace(",", ".")
        if risposta == "":
            return None
        try:
            percentuale = float(risposta)
        except ValueError:
            print(f"'{risposta}' non è un numero. Scrivi per esempio 80")
            continue
        if 0 <= percentuale <= 100:
            return percentuale
        print("La percentuale deve essere compresa tra 0 e 100")


def chiedi_filtri(archivio):
    """Chiede all'utente i criteri del filtro, uno alla volta, e li restituisce in una tupla."""
    tipo = chiedi_scelta("Tipo", archivio.tipi_ammessi)
    canale = chiedi_scelta("Canale", archivio.canali_ammessi)
    tema = chiedi_scelta("Tema", archivio.temi())
    min_donne = chiedi_percentuale()
    return tipo, canale, tema, min_donne


def menu_utente(archivio):
    """Menu interattivo per consultare l'archivio: visualizzazione, ricerca, filtri e statistiche."""
    print("Benvenuta, benvenuto nell'archivio di Voci!")
    while True:
        print("\nCosa vuoi fare?")
        print("  1. Mostra l'archivio")
        print("  2. Cerca una parola")
        print("  3. Filtra per più criteri")
        print("  4. Statistiche dell'archivio")
        print("  0. Esci")
        scelta = input("Scrivi il numero dell'opzione: ").strip()

        if scelta == "1":
            archivio.mostra()

        elif scelta == "2":
            parola = input("Parola da cercare: ")
            try:
                risultati = archivio.cerca(parola)
                print(f"\nRisultati trovati: {len(risultati)}")
                archivio.mostra(risultati)
            except ValueError as errore:
                print(f"Errore: {errore}")

        elif scelta == "3":
            tipo, canale, tema, min_donne = chiedi_filtri(archivio)
            if tipo is None and canale is None and tema is None and min_donne is None:
                print("\nNessun criterio inserito: ecco tutto l'archivio.")
            risultati = archivio.filtra(tipo, canale, tema, min_donne)
            print(f"\nRisultati trovati: {len(risultati)}")
            archivio.mostra(risultati)

        elif scelta == "4":
            archivio.statistiche()

        elif scelta == "0":
            print("Grazie per aver consultato l'archivio di Voci!")
            break

        else:
            print(f"'{scelta}' non è un'opzione valida: scrivi un numero da 0 a 4")

**Per avviare il menu** togli il `#` dalla riga qui sotto ed esegui la cella: è commentata perché, aspettando le risposte dell'utente, bloccherebbe l'esecuzione completa del notebook.

In [14]:
menu_utente(archivio)

Benvenuta, benvenuto nell'archivio di Voci!

Cosa vuoi fare?
  1. Mostra l'archivio
  2. Cerca una parola
  3. Filtra per più criteri
  4. Statistiche dell'archivio
  0. Esci


Scrivi il numero dell'opzione:  1


[intervista] Madri e giornaliste: il rientro dopo il congedo | lavoro e maternità | youtube | 2026-09-14 | voci femminili 100% (donne 3, uomini 0) | con Chiara Lodi
[video] La piramide delle redazioni | leadership | youtube | 2026-09-21 | voci femminili 80% (donne 4, uomini 1) | durata 12 min
[articolo] Chi firma le notizie in Italia | nuovi media | blog | 2026-09-28 | voci femminili 83% (donne 5, uomini 1) | di Marta Rinaldi
[intervista] Fondare un media a trent'anni | nuovi media | youtube | 2026-10-05 | voci femminili 100% (donne 2, uomini 0) | con Sara Monti
[video] Allenatrici in serie A | sport | social | 2026-10-12 | voci femminili 40% (donne 2, uomini 3) | durata 4 min
[video] Le esperte che i media non chiamano | scienza e tecnologia | youtube | 2026-10-22 | voci femminili 100% (donne 6, uomini 0) | durata 15 min
[intervista] Mobilità e città a misura di donna | politica | youtube | 2026-10-23 | voci femminili 50% (donne 1, uomini 1) | con Anna Bruni

Cosa vuoi fare?
  1. Most

Scrivi il numero dell'opzione:  2
Parola da cercare:  piramide



Risultati trovati: 1
[video] La piramide delle redazioni | leadership | youtube | 2026-09-21 | voci femminili 80% (donne 4, uomini 1) | durata 12 min

Cosa vuoi fare?
  1. Mostra l'archivio
  2. Cerca una parola
  3. Filtra per più criteri
  4. Statistiche dell'archivio
  0. Esci


...


Risultati trovati: 0
Nessun contenuto da mostrare.

Cosa vuoi fare?
  1. Mostra l'archivio
  2. Cerca una parola
  3. Filtra per più criteri
  4. Statistiche dell'archivio
  0. Esci


Scrivi il numero dell'opzione:  4


Contenuti in archivio: 7
Voci femminili nell'archivio: 79.3% (esperte nei media digitali: 39%)

Contenuti per tipo:
  intervista: 43%
  video: 43%
  articolo: 14%

Tema più trattato: nuovi media (2 contenuti)
Contenuto più recente: 'Mobilità e città a misura di donna' (2026-10-23)
Tema su cui Voci deve impegnarsi di più: sport (40% di voci femminili)

Cosa vuoi fare?
  1. Mostra l'archivio
  2. Cerca una parola
  3. Filtra per più criteri
  4. Statistiche dell'archivio
  0. Esci


Scrivi il numero dell'opzione:  0


Grazie per aver consultato l'archivio di Voci!


## 7. Fonti e note

- **Voci** è l'azienda ipotetica della consegna del progetto; contenuti, titoli e persone dell'archivio sono inventati.
- **Esperte citate e firme nei media digitali (39% e 29%)**: Osservatorio di Pavia, *Global Media Monitoring Project (GMMP) 2025 – Italia*. Il GMMP analizza una sola giornata di notizie, quindi il dato è indicativo. https://www.osservatorio.it/wp-content/uploads/2016/03/GMMP25-_IT_EN.pdf
- Alcuni dati e idee riprendono il mio progetto del capitolo **"Introduzione alla Data Science e AI"** di start2impact, dedicato alle donne nel settore dell'informazione italiano, e l'**Obiettivo 5 dell'Agenda ONU 2030 – Parità di genere**. https://unric.org/it/obiettivo-5-raggiungere-luguaglianza-di-genere-ed-emancipare-tutte-le-donne-e-le-ragazze/